In [1]:
!pip install -q plotnine

StatementMeta(, a65225a5-b18d-402d-b914-0a8fc71bc9c4, 3, Finished, Available, Finished)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pyspark.sql.functions import col
from plotnine import (
    ggplot, aes, geom_boxplot, geom_jitter, geom_bar, geom_text, labs, theme, element_text,
    element_rect, scale_color_identity, scale_fill_discrete, theme_minimal, ylim
)


StatementMeta(, a65225a5-b18d-402d-b914-0a8fc71bc9c4, 4, Finished, Available, Finished)

In [6]:
# ================================ RANGE DATETIME ================================ 
start_date = "2024-01-01 00:00:00"
end_date = "2024-03-21 00:00:00"
# ================================== EQUIPMENT ================================== 
data_equipments = {
    "Equipment": [
        "TRH380", "TRH342", "TRH343", "TRH361", "TRH362", "TRH363", "TRH364",
        "TRH374", "TRH376", "TRH377", "TRH378", "TRH379", "TRH382", "TRH385",
        "TRH386", "TRH390", "TRH391", "TRH392", "TRH394", "TRH395", "TRH396",
        "TRH398", "TRH340", "TRH344", "TRH345", "TRH346", "TRH359", "TRH360",
        "TRH365", "TRH366", "TRH367", "TRH368", "TRH369", "TRH370", "TRH371",
        "TRH372", "TRH373", "TRH375", "TRH381", "TRH383", "TRH384", "TRH387",
        "TRH388", "TRH389", "TRH393", "TRH399", "TRH347", "TRH400", "TRH401",
        "TRH402", "TRH403", "TRH404", "TRH405"
    ],
    "ClassType": [
        "Cat 793C", "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793D",
        "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C",
        "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C",
        "Cat 793C", "Cat 793C", "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793D",
        "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793D", "Cat 793C", "Cat 793C",
        "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C",
        "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793C", "Cat 793D", "Cat 793F", "Cat 793F",
        "Cat 793F", "Cat 793F", "Cat 793F", "Cat 793F"
    ]
}

# Create DataFrame
df_source_equipments = pd.DataFrame(data_equipments)

# Group by 'class_type' and aggregate equipment into lists
df_equipments = df_source_equipments.groupby("ClassType")["Equipment"].apply(list).reset_index()
df_equipments.columns = ["ClassType", "Equipment"]

# ================================== MEASUREMENTS ==================================
data_measurements = {
    "Metricas": [
        "Ground Speed",
        "Lt Front Brake Temperature",
        "Lt Rear Brake Temperature",
        "Rt Front Brake Temperature",
        "Rt Rear Brake Temperature",
        "Rtf-Ltf Brake Temperature",
        "Rtr-Ltr Brake Temperature",
    ],
    "MinLimit": [0, 0, 0, 0, 0, 0, 0],
    "MaxLimit": [30, 1000, 1000, 1000, 1000, 50, 50],
}
"""
data_measurements = {
    "Metricas": [
        "Ground Speed",
        "Lt Front Brake Temperature",
        "Lt Rear Brake Temperature",
        "Rt Front Brake Temperature",
        "Rt Rear Brake Temperature",
        "Rtf-Ltf Brake Temperature",
        "Rtr-Ltr Brake Temperature",
    ],
    "MinLimit": [0, 0, 0, 0, 0, 0, 0],
    "MaxLimit": [30, 1000, 1000, 1000, 1000, 50, 50],
}
data_measurements = {
    "Metricas": [
        "Payload",
        "Rt F Susp Cyl",
        "Rt R Susp Cyl",
        "Lt F Susp Cyl",
        "Lt R Susp Cyl",
    ],
    "MinLimit": [80, 0, 0, 0, 0],
    "MaxLimit": [300, 1000, 1000, 1000, 1000],
}
data_measurements = {
    "Metricas": [
        "Boost Pressure",
        "Engine Speed",
        "Ground Speed",
        "Air Filter",
        "Engine Cool Temperature",
    ],
    "MinLimit": [-100, 0, 0, -5.0, 30],
    "MaxLimit": [1000, 1000, 50, 8.0, 120],
}
"""

df_measurements = pd.DataFrame(data_measurements)

StatementMeta(, a65225a5-b18d-402d-b914-0a8fc71bc9c4, 8, Finished, Available, Finished)

In [4]:
source_df = spark.table("sensores_camiones")

StatementMeta(, a65225a5-b18d-402d-b914-0a8fc71bc9c4, 9, Finished, Available, Finished)

In [7]:
# Generate a timestamp (e.g., 2025-03-19_14-30-45)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# Iterate through DataFrame and print each row
for index, row_measure in df_measurements.iterrows():
    print(f"Metric: {row_measure['Metricas']}, MinLimit: {row_measure['MinLimit']}, MaxLimit: {row_measure['MaxLimit']}")
    for _, row_equipment in df_equipments.iterrows():
        print(f"Class Type: {row_equipment['ClassType']}")
        df_filtered = source_df.filter(
            (col("Nombre_Camion").isin(row_equipment["Equipment"])) & 
            (col("Nombre_Tag") == row_measure['Metricas']) & 
            (col("Fecha_Hora").between(start_date, end_date)) 
        )
        source_pandas_df = df_filtered.toPandas()
        dataset = source_pandas_df[["Nombre_Camion", "Valor"]]
        dataset.columns = ["Equipment", "Value"]
        original_dataset = dataset.copy()
        # Remove NaN values and filter out values 
        dataset = dataset.dropna(subset=["Value"])  # Remove NaN
        dataset = dataset[(dataset["Value"] >= row_measure['MinLimit']) & (dataset["Value"] <= row_measure['MaxLimit'])]  # Keep values
        # Compute summary statistics per Equipment (like the R group_by + summarise)
        row_count = len(dataset)
        print(f"Total rows: {row_count}")
        if row_count > 0:
            data_barchart = original_dataset.groupby("Equipment").size().reset_index(name="Count")
            data_boxplot = dataset.groupby("Equipment").agg(
                y0=("Value", "min"),
                y25=("Value", lambda x: np.percentile(x, 25)),
                y50=("Value", "median"),
                y75=("Value", lambda x: np.percentile(x, 75)),
                y100=("Value", "max"),
                mean=("Value", "mean")
            ).reset_index()
            # Merge summary stats back into dataset
            dataset_merged = pd.merge(dataset, data_boxplot, on="Equipment", how="left")

            # Create a 'color' column based on conditions (nested np.where replicates ifelse)
            dataset_merged['color'] = np.where(
                dataset_merged['Value'] >= dataset_merged['y100'], "#000000",
                np.where(
                    dataset_merged['Value'] >= dataset_merged['y50'], "#ac0000",
                    "#000065"  # Note: the last two conditions in the R code are identical.
                )
            )

            # Reorder Equipment factor based on median Value in descending order.
            median_order = dataset.groupby("Equipment")["Value"].median().sort_values(ascending=False).index.tolist()
            dataset_merged['Equipment'] = pd.Categorical(dataset_merged['Equipment'], categories=median_order, ordered=True)
            # Create the plot similar to the ggplot2 code.
            boxplot = (ggplot(dataset_merged, aes(x='Equipment', y='Value'))
                    + geom_jitter(aes(color="color"), alpha=0.5, size=2, width=0.25) 
                    + scale_color_identity()
                    + geom_boxplot(alpha=0.25, fill="white", outlier_shape=None)
                    + labs(title=f"{row_measure['Metricas']} - {row_equipment['ClassType']} ({start_date} - {end_time})", x="Equipment", y="Value")
                    + theme(
                            axis_text_x=element_text(angle=90, vjust=1, ha="right"),
                            panel_background=element_rect(fill="white"),
                            legend_position="none"
                    )
            )
            #plot.draw()
            boxplot.save(f"/lakehouse/default/Files/Images/Boxplot_{row_measure['Metricas']}_{row_equipment['ClassType']}_{timestamp}.png", width=16, height=9, dpi=400)

            max_count = data_barchart["Count"].max()
            barchart = (
                ggplot(data_barchart, aes(x="Equipment", y="Count")) +
                geom_bar(stat="identity", fill="navy") +  # Create bars
                geom_text(aes(label="Count"), va='bottom', size=10) +  # Add labels above bars
                ylim(0, max_count * 1.5) + 
                theme_minimal() +
                labs(title=f"{row_measure['Metricas']} - {row_equipment['ClassType']} ({start_date} - {end_time})", x="Haul Truck", y="Sensor Count") + 
                theme(
                    axis_text_x=element_text(angle=90, vjust=1),
                    plot_background=element_rect(fill='white', color='white'),
                    panel_background=element_rect(fill="white"),
                    legend_position="none"
                )
            )
            barchart.save(f"/lakehouse/default/Files/Images/Barchart_{row_measure['Metricas']}_{row_equipment['ClassType']}_{timestamp}.png", width=16, height=9, dpi=400)


StatementMeta(, a65225a5-b18d-402d-b914-0a8fc71bc9c4, 10, Finished, Available, Finished)

Metric: Ground Speed, MinLimit: 0, MaxLimit: 30
Class Type: Cat 793C


Total rows: 3422492


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 16 x 9 in image.
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /lakehouse/default/Files/Images/Boxplot_Ground Speed_Cat 793C_2025-03-21_22-19-09.png


Class Type: Cat 793D


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 16 x 9 in image.
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /lakehouse/default/Files/Images/Boxplot_Ground Speed_Cat 793D_2025-03-21_22-19-09.png


... (33 outputs left out)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 16 x 9 in image.
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /lakehouse/default/Files/Images/Boxplot_Rtr-Ltr Brake Temperature_Cat 793C_2025-03-21_22-19-09.png


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 16 x 9 in image.
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /lakehouse/default/Files/Images/Barchart_Rtr-Ltr Brake Temperature_Cat 793C_2025-03-21_22-19-09.png


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 16 x 9 in image.
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /lakehouse/default/Files/Images/Boxplot_Rtr-Ltr Brake Temperature_Cat 793D_2025-03-21_22-19-09.png


Class Type: Cat 793F
Total rows: 0


### Code to delete files

In [32]:
import glob
import os

folder_path = "/lakehouse/default/Files/Images/*"  # Add /* to target all files
for file in glob.glob(folder_path):
    if os.path.isfile(file):  # Ensure it's a file, not a directory
        os.remove(file)  # Delete the file

StatementMeta(, a65225a5-b18d-402d-b914-0a8fc71bc9c4, 11, Finished, Available, Finished)